# One patient, every modality: a showcase of the asteroidea SDK

This notebook is a best-case, single-patient analysis built only from the `asteroidea` SDK. It follows
patient **CS62** of the movie-watching study (Keles et al. 2024, *Sci Data* 11:214,
doi [10.1038/s41597-024-03029-1](https://doi.org/10.1038/s41597-024-03029-1)) through both of their
recording runs, `ses-P62CSR1` and `ses-P62CSR2` of DANDI 000623, and through their structural MRI in the
linked OpenNeuro dataset, where the same person is `sub-p62cs`.

The patient watched an 8-minute movie clip, then saw 40 still frames, half from the clip ("old") and
half not ("new"), and rated each from 1 (sure new) to 6 (sure old). Microwires in five brain areas
recorded single neurons and local field potentials (LFP) throughout.

What the notebook does, section by section:

1. **Setup**: sign in, one cell of constants, small helper functions.
2. **Who and what**: a metadata card from the catalog: files, units, contacts, LFP series, the task.
3. **Anatomy**: the patient's own T1 MRI, and the electrode contacts on a standard brain.
4. **Session overview**: every neuron over the whole session, and how well the patient remembered.
5. **Scene cuts** (the paper's Fig 7): neurons that tell a change of scene from a continuity cut.
6. **Recognition** (the paper's Fig 6): neurons that tell new frames from old ones.
7. **Tuning**: firing rate against confidence, cut type and time in the movie.
8. **LFP**: spectrograms per area, event-locked spectrograms and the paper's high-frequency power.
9. **Summary**: the headline numbers in one table.

Everything is computed live from the platform's catalog and the raw recordings, with polars for
tables and numpy and scipy for the signal work. A full run takes about 15 minutes; most of it is
reading the two runs' LFP.

## 1. Setup

The imports: the SDK, polars, numpy, scipy, matplotlib, nilearn (for the standard brain) and
IPython's `display`, imported as `show`. `asteroidea.sign_in()` prints a login address the first time
and reuses the saved sign-in after that.

In [ ]:
import contextlib
import itertools
import json
import tempfile
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import display as show
from matplotlib.lines import Line2D
from nilearn import datasets
from nilearn.plotting.image import plot_anat, plot_glass_brain
from scipy import fft, signal, stats

import asteroidea

asteroidea.sign_in()

One cell holds every choice the analysis makes: the patient, the two runs and the paper's parameters
(windows, bin size, bootstrap size and seed, the HFB band). Change them here and nothing else.
`SAVE_RESULTS` turns the last section's saves on or off, and `RESULTS_RUN` names the run they are
saved under.


In [ ]:
DATASET = "dandi-000623"
SUBJECT = "sub-CS62"
MRI_SUBJECT = "sub-p62cs"
RUNS = ("ses-P62CSR1", "ses-P62CSR2")
RUN_LABEL = {"ses-P62CSR1": "R1", "ses-P62CSR2": "R2"}

# Section 10 saves the figures, the headline numbers and the key tables as results
# of the run RESULTS_RUN; set SAVE_RESULTS to False to save nothing.
SAVE_RESULTS = True
RESULTS_RUN = "showcase-cs62"

# Paper parameters (Keles et al. 2024, release code ephys_qc)
PSTH_BIN_S = 0.25
CUT_WINDOW_S = (0.0, 1.0)
CUT_PLOT_S = (-0.5, 1.0)
PROBE_WINDOW_S = (0.2, 1.7)
PROBE_PLOT_S = (-1.0, 2.0)
BOOTSTRAP_N = 10_000
SEED = 13
ALPHA = 0.05

# Session overview and LFP
OVERVIEW_BIN_S = 5.0
OVERVIEW_Z_CLIP = 3.0
MIN_EXAMPLE_RATE_HZ = 0.5
N_EXAMPLES = 3
MIN_BUNDLE_WIRES = 2
AXIAL_BELOW_TOP_MM = 90.0
ARTIFACT_SD = 6.0
MAD_TO_SD = 1.4826
ARTIFACT_PAD_S = 0.25
MIN_CLEAN_EVENTS = 5
WHITE_DB = 6.0
MICRO_SERIES = "/processing/ecephys/LFP_micro/ElectricalSeries"
N_MICRO = 56
LFP_READ_S = 3600.0  # longer than any run: the read stops at the end of the series
LINE_HZ = 60.0
HIGHPASS_HZ = 0.1
HFB_EDGES_HZ = tuple(range(70, 180, 10))  # 70-80, ..., 160-170 Hz
SPEC_WINDOW_S = 2.0
SPEC_MAX_HZ = 150.0
EPOCH_S = (-1.0, 2.0)
BASELINE_S = (-0.5, 0.0)
EPOCH_NPERSEG = 256
EPOCH_STEP = 25
EPOCH_MIN_HZ = 2.0
HFB_RATE_HZ = 100
MIN_SAMPLES = 2

AREAS = ("amygdala", "hippocampus", "preSMA", "ACC", "vmPFC")
AREA_COLOR = {
    "amygdala": "#d55e00",
    "hippocampus": "#009e73",
    "preSMA": "#0072b2",
    "ACC": "#cc79a7",
    "vmPFC": "#e69f00",
    "other": "#7f7f7f",
}
CONDITION_COLOR = {
    "change": "#c0392b",
    "continuity": "#7f8c8d",
    "new": "#2471a3",
    "old": "#d68910",
}

pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(120)
pl.Config.set_tbl_width_chars(160)
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.frameon": False,
        "font.size": 9,
    }
)

Every read from here to the end is recorded in one **run**, `asteroidea.run`, opened here and closed
in section 10. The run is what makes the saved results traceable: each saved file carries the
catalog tables, files and analysis settings this notebook read, and the parameters above. Figures
are kept in `FIGURES` as they are drawn, for section 10.

In [ ]:
# Running this cell again closes the run it opened before: runs do not nest.
if (previous := globals().get("run_scope")) is not None:
    previous.close()
run_scope = contextlib.ExitStack()
results_run = run_scope.enter_context(
    asteroidea.run(
        RESULTS_RUN,
        {
            "dataset": DATASET,
            "subject": SUBJECT,
            "runs": list(RUNS),
            "psth_bin_s": PSTH_BIN_S,
            "cut_window_s": list(CUT_WINDOW_S),
            "probe_window_s": list(PROBE_WINDOW_S),
            "bootstrap_n": BOOTSTRAP_N,
            "seed": SEED,
            "alpha": ALPHA,
        },
    )
)
FIGURES: dict[str, Any] = {}

Small helpers, each doing one thing. The test helpers mirror the paper's release code: a cell's rate in
a fixed window per trial, compared between two conditions with a bootstrap of the difference of means
(10,000 resamples, seed 13, two-sided p). The recognition test resamples both sides to the smaller
group's size, as the release code does.

In [ ]:
def area_of(location: str | None) -> str:
    """Return the short area name of an electrode or unit location."""
    text = (location or "").lower()
    names = {
        "amyg": "amygdala",
        "hippo": "hippocampus",
        "presma": "preSMA",
        "pre-sma": "preSMA",
        "supplementary": "preSMA",
        "vmpfc": "vmPFC",
        "orbito": "vmPFC",
        "ofc": "vmPFC",
        "acc": "ACC",
        "cingul": "ACC",
    }
    return next((area for key, area in names.items() if key in text), "other")


def window_rates(
    spikes: np.ndarray, onsets: np.ndarray, window: tuple[float, float]
) -> np.ndarray:
    """Return the firing rate (Hz) of one unit in `window` after each onset."""
    lo = np.searchsorted(spikes, onsets + window[0], side="left")
    hi = np.searchsorted(spikes, onsets + window[1], side="left")
    return (hi - lo) / (window[1] - window[0])


def bootstrap_p(a: np.ndarray, b: np.ndarray, *, balanced: bool) -> float:
    """Return the two-sided bootstrap p of mean(a) - mean(b) (release-code method)."""
    rng = np.random.default_rng(SEED)
    size_a, size_b = (min(a.size, b.size),) * 2 if balanced else (a.size, b.size)
    diffs = a[rng.integers(0, a.size, (BOOTSTRAP_N, size_a))].mean(axis=1) - b[
        rng.integers(0, b.size, (BOOTSTRAP_N, size_b))
    ].mean(axis=1)
    low = (1 + np.count_nonzero(diffs <= 0)) / (BOOTSTRAP_N + 1)
    high = (1 + np.count_nonzero(diffs >= 0)) / (BOOTSTRAP_N + 1)
    return min(1.0, 2 * min(low, high))


def trial_psth(
    spikes: np.ndarray, onsets: np.ndarray, window: tuple[float, float]
) -> np.ndarray:
    """Return the rate (Hz) per trial and PSTH_BIN_S bin, shape (trials, bins)."""
    edges = np.arange(window[0], window[1] + PSTH_BIN_S / 2, PSTH_BIN_S)
    return np.stack(
        [np.histogram(spikes - onset, edges)[0] / PSTH_BIN_S for onset in onsets]
    )


def bin_centres(window: tuple[float, float]) -> np.ndarray:
    """Return the centres of the PSTH bins of `window`."""
    edges = np.arange(window[0], window[1] + PSTH_BIN_S / 2, PSTH_BIN_S)
    return (edges[:-1] + edges[1:]) / 2


def mean_sem(rows: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return the mean and standard error over the first axis."""
    count = max(rows.shape[0], 1)
    return rows.mean(axis=0), rows.std(axis=0, ddof=1) / np.sqrt(count)


def draw_raster_psth(
    axes: tuple[Any, Any],
    spikes: np.ndarray,
    groups: dict[str, np.ndarray],
    window: tuple[float, float],
) -> None:
    """Draw a raster (trials grouped by condition) above its PSTH with SEM bands."""
    raster, curve = axes
    row = 0
    for block, (name, onsets) in enumerate(groups.items()):
        trains = [
            spikes[(spikes >= t + window[0]) & (spikes < t + window[1])] - t
            for t in onsets
        ]
        raster.eventplot(
            trains,
            lineoffsets=np.arange(row, row + len(trains)),
            linelengths=1.0,
            linewidths=1.3,
            colors=CONDITION_COLOR[name],
        )
        if block % 2:
            raster.axhspan(
                row - 0.5, row + len(trains) - 0.5, color="#f2f2f2", zorder=0
            )
        if row:
            raster.axhline(row - 0.5, color="black", linewidth=0.6)
        row += len(trains)
        mean, sem = mean_sem(trial_psth(spikes, onsets, window))
        centres = bin_centres(window)
        curve.plot(
            centres, mean, color=CONDITION_COLOR[name], label=f"{name} ({len(onsets)})"
        )
        curve.fill_between(
            centres, mean - sem, mean + sem, color=CONDITION_COLOR[name], alpha=0.25
        )
    for ax in axes:
        ax.axvline(0, color="black", linewidth=0.7, linestyle="--")
        ax.set_xlim(*window)
    raster.set_ylim(-0.5, row - 0.5)
    raster.set_ylabel("trial")
    raster.set_xticklabels([])
    curve.set_xlabel("time from onset (s)")
    curve.set_ylabel("rate (Hz)")
    curve.legend(fontsize=7)


def roc_auc(
    ratings: np.ndarray, is_old: np.ndarray
) -> tuple[np.ndarray, np.ndarray, float]:
    """Return the ROC (false alarm and hit rates per rating criterion) and its AUC."""
    old, new = ratings[is_old], ratings[~is_old]
    criteria = np.arange(6, 0, -1)
    hits = np.array([0.0, *[(old >= c).mean() for c in criteria]])
    false_alarms = np.array([0.0, *[(new >= c).mean() for c in criteria]])
    greater = (old[:, None] > new[None, :]).mean()
    ties = (old[:, None] == new[None, :]).mean()
    return false_alarms, hits, float(greater + ties / 2)


def zscore_rows(rows: np.ndarray) -> np.ndarray:
    """Return each row z-scored; a row with no spread becomes NaN."""
    spread = rows.std(axis=1, keepdims=True)
    with np.errstate(invalid="ignore", divide="ignore"):
        return (rows - rows.mean(axis=1, keepdims=True)) / np.where(
            spread > 0, spread, np.nan
        )


def require(frame: pl.DataFrame | pl.Series, what: str) -> pl.DataFrame:
    """Return `frame` as a table, or raise naming `what` when it is empty."""
    if isinstance(frame, pl.Series):
        msg = f"{what}: expected a table, got a single column"
        raise TypeError(msg)
    if frame.is_empty():
        msg = f"no rows for {what}: the section cannot run without them"
        raise LookupError(msg)
    return frame

Signal helpers for section 8. The paper's LFP pipeline (release code `prep_filterLFP.py`) notches 60 Hz
and its harmonics, high-passes at 0.1 Hz and re-references to the common average. It then takes Morlet
power from 70 to 170 Hz in 10 Hz steps, z-scores each frequency over time and averages them: the
high-frequency broadband (HFB) signal. Here scipy replaces MNE: zero-phase IIR notches and a Butterworth
high-pass, and band power from the Hilbert envelope of each 10 Hz band instead of a Morlet transform.
The common average is taken within each microwire bundle.

In [ ]:
def clean_bundle(raw: np.ndarray, rate: float) -> np.ndarray:
    """Return a bundle (channels, samples) notched, high-passed and re-referenced."""
    out = raw.astype(np.float64)
    for harmonic in np.arange(LINE_HZ, rate / 2, LINE_HZ):
        b, a = signal.iirnotch(harmonic, Q=30.0, fs=rate)
        out = signal.filtfilt(b, a, out, axis=-1)
    sos = signal.butter(2, HIGHPASS_HZ, btype="highpass", fs=rate, output="sos")
    out = signal.sosfiltfilt(sos, out, axis=-1)
    return (out - out.mean(axis=0, keepdims=True)).astype(np.float32)


def hfb_trace(clean: np.ndarray, rate: float) -> np.ndarray:
    """Return the HFB trace of clean channels: one value per sample, channel mean."""
    n = clean.shape[-1]
    total = np.zeros(n)
    for lo, hi in itertools.pairwise(HFB_EDGES_HZ):
        sos = signal.butter(4, (lo, hi), btype="bandpass", fs=rate, output="sos")
        band = signal.sosfiltfilt(sos, clean, axis=-1)
        power = (
            np.abs(signal.hilbert(band, N=fft.next_fast_len(n), axis=-1)[:, :n]) ** 2
        )
        total += zscore_rows(power).mean(axis=0)
    return total / (len(HFB_EDGES_HZ) - 1)


def downsample(trace: np.ndarray, factor: int) -> np.ndarray:
    """Return the block means of `trace` over `factor` samples."""
    usable = trace.size // factor * factor
    return trace[:usable].reshape(-1, factor).mean(axis=1)


def epoch_power(
    clean: np.ndarray, start_s: float, rate: float, onsets: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return frequencies, times and power (events, freqs, times) around each onset.

    The power is averaged over the bundle's channels. `start_s` is the trials-clock time
    of the first sample.
    """
    pad = EPOCH_NPERSEG / 2 / rate
    first = np.round((onsets + EPOCH_S[0] - pad - start_s) * rate).astype(int)
    width = round((EPOCH_S[1] - EPOCH_S[0] + 2 * pad) * rate)
    inside = (first >= 0) & (first + width <= clean.shape[-1])
    windows = np.zeros((0, clean.shape[0], width), dtype=clean.dtype)
    if np.any(inside):
        windows = np.stack([clean[:, f : f + width] for f in first[inside]])
    freqs, times, power = signal.spectrogram(
        windows,
        fs=rate,
        nperseg=EPOCH_NPERSEG,
        noverlap=EPOCH_NPERSEG - EPOCH_STEP,
        axis=-1,
    )
    return freqs, times + EPOCH_S[0] - pad, power.mean(axis=1)


def check_power(name: str, values: np.ndarray, *, positive: bool) -> np.ndarray:
    """Return `values`, or raise when any is not finite (or not positive, for power)."""
    if values.size == 0 or not np.all(np.isfinite(values)):
        msg = f"{name}: {values.size} values, some empty, NaN or infinite"
        raise ValueError(msg)
    if positive and not np.all(values > 0):
        msg = (
            f"{name}: power is zero somewhere (min {values.min():.3g}); "
            "the reference removed the whole signal"
        )
        raise ValueError(msg)
    return values


def db_change(power: np.ndarray, times: np.ndarray) -> np.ndarray:
    """Return the median power over events as dB change from the BASELINE_S window."""
    middle = np.median(power, axis=0)
    base = middle[:, (times >= BASELINE_S[0]) & (times < BASELINE_S[1])].mean(axis=1)
    return 10 * np.log10(middle / base[:, None])


def artifact_mask(clean: np.ndarray, hfb: np.ndarray, rate: float) -> np.ndarray:
    """Return a per-sample artifact flag, padded by ARTIFACT_PAD_S on each side.

    A sample is flagged when any wire is beyond ARTIFACT_SD robust SDs (median
    absolute deviation x 1.4826) of its own median, or the HFB z is beyond ARTIFACT_SD.
    """
    centre = np.median(clean, axis=1, keepdims=True)
    robust = np.median(np.abs(clean - centre), axis=1, keepdims=True) * MAD_TO_SD
    hit = (np.abs(clean - centre) > ARTIFACT_SD * robust).any(axis=0) | (
        hfb > ARTIFACT_SD
    )
    pad = round(ARTIFACT_PAD_S * rate)
    total = np.concatenate([[0], np.cumsum(hit)])
    index = np.arange(hit.size)
    lo = np.clip(index - pad, 0, hit.size)
    hi = np.clip(index + pad + 1, 0, hit.size)
    return (total[hi] - total[lo]) > 0


def clean_onsets(
    onsets: np.ndarray,
    window: tuple[float, float],
    mask: np.ndarray,
    start: tuple[float, float],
) -> np.ndarray:
    """Return the onsets whose window does not overlap a flagged sample.

    `start` is (trials-clock time of the first sample, sample rate).
    """
    first_s, rate = start
    total = np.concatenate([[0], np.cumsum(mask)])
    lo = np.clip(
        np.floor((onsets + window[0] - first_s) * rate).astype(int), 0, mask.size
    )
    hi = np.clip(
        np.ceil((onsets + window[1] - first_s) * rate).astype(int), 0, mask.size
    )
    return onsets[(total[hi] - total[lo]) == 0]

## 2. Who and what

Everything here comes from the catalog through `asteroidea.sql()`, which returns an Arrow table that
`pl.from_arrow` turns into polars. First the person: the **link** that joins the DANDI label `sub-CS62`
and the OpenNeuro label `sub-p62cs` under one participant key, then every file of both.

In [ ]:
links = require(
    pl.from_arrow(asteroidea.links(study="movie-watching")), "the study's links"
)
patient_key = require(
    links.filter(pl.col("subject") == SUBJECT), f"a subject link of {SUBJECT}"
)["subject_key"][0]
show(
    links.filter(pl.col("subject_key") == patient_key).select(
        "subject_key", "dataset", "subject", "status", "linked_by", "basis"
    )
)
files = require(
    pl.from_arrow(
        asteroidea.sql(
            "SELECT dataset, subject, bids_session, datatype, suffix, s3_key, "
            "size_bytes "
            "FROM files WHERE (dataset = 'dandi-000623' AND subject = 'sub-CS62') "
            "OR subject = 'sub-p62cs' ORDER BY dataset, s3_key"
        )
    ),
    "the files of CS62",
)
show(
    files.select(
        "dataset",
        "subject",
        "bids_session",
        "datatype",
        pl.col("s3_key").str.split("/").list.last().alias("file"),
        (pl.col("size_bytes") / 1e6).round(1).alias("size_mb"),
    )
)

Next the two runs. `load_session()` reads one run's units, spike times, events and LFP series list
(the LFP itself is read later, on demand). Units were spike-sorted per run, so a unit in R1 and a unit
in R2 are different cells as far as the data can tell. Every per-unit test below uses its own run's
trials, and the population summaries pool the units of both runs.

In [ ]:
started = time.monotonic()
sessions = {run: asteroidea.load_session(DATASET, SUBJECT, run) for run in RUNS}
units = require(
    pl.concat(
        [
            s.units.select(
                pl.lit(RUN_LABEL[run]).alias("run"),
                "unit_index",
                "electrode_index",
                "location",
                "spike_count",
                "mean_rate_hz",
                "columns_json",
            )
            for run, s in sessions.items()
        ]
    ).with_columns(
        pl.col("location").map_elements(area_of, return_dtype=pl.String).alias("area")
    ),
    "the units of both runs",
)
show(
    pl.DataFrame(
        {
            "sessions loaded": [len(sessions)],
            "seconds": [round(time.monotonic() - started, 1)],
        }
    )
)
show(
    units.pivot(on="run", index="area", values="unit_index", aggregate_function="len")
    .fill_null(0)
    .sort("area")
)

The electrode contacts. Each run's NWB file lists 96: 56 microwires (indices 0 to 55) and 40 macro
contacts (56 to 95), in five areas plus a set labelled RSPE. The x, y, z columns are MNI coordinates
(MNI152NLin2009cAsym, in mm, as the paper states). The RSPE contacts and six macro contacts have no
coordinates; they are counted here and left out of the anatomy plots.

In [ ]:
electrodes = require(
    pl.from_arrow(
        asteroidea.sql(
            "SELECT bids_session, electrode_index, location, group_name, x, y, z, "
            "columns_json "
            "FROM electrodes WHERE dataset = 'dandi-000623' AND subject = 'sub-CS62' "
            "ORDER BY bids_session, electrode_index"
        )
    ),
    "the electrodes of CS62",
).with_columns(
    pl.col("location").map_elements(area_of, return_dtype=pl.String).alias("area"),
    pl.when(pl.col("electrode_index") < N_MICRO)
    .then(pl.lit("micro"))
    .otherwise(pl.lit("macro"))
    .alias("kind"),
    pl.col("columns_json").str.json_path_match("$.origchannel_name").alias("channel"),
)
show(
    electrodes.group_by("bids_session", "location", "kind")
    .agg(
        pl.len().alias("contacts"),
        pl.col("x").is_not_null().sum().alias("with_xyz"),
    )
    .sort("bids_session", "location", "kind")
)

The LFP series of each run, as the catalog lists them (no samples are read here), and a compact view of
the spike-sorting quality the dataset ships per unit: the mean signal-to-noise ratio and the
percentage of inter-spike intervals under 3 ms (`isibelow`).

In [ ]:
show(
    pl.concat(
        [
            s.series.filter(pl.col("path").str.contains("LFP")).select(
                pl.lit(RUN_LABEL[run]).alias("run"),
                "path",
                pl.col("shape")
                .cast(pl.List(pl.String))
                .list.join(" x ")
                .alias("shape"),
                "rate_hz",
            )
            for run, s in sessions.items()
        ]
    )
)
quality = units.with_columns(
    pl.col("columns_json")
    .str.json_path_match("$.meanSNR")
    .cast(pl.Float64)
    .alias("mean_snr"),
    pl.col("columns_json")
    .str.json_path_match("$.isibelow")
    .cast(pl.Float64)
    .alias("isi_below"),
)
show(
    quality.group_by("run", "area")
    .agg(
        pl.len().alias("units"),
        pl.col("mean_rate_hz").median().round(2).alias("median_rate_hz"),
        pl.col("mean_snr").median().round(2).alias("median_snr"),
        pl.col("isi_below").median().round(2).alias("median_isi_below"),
    )
    .sort("run", "area")
)

The task, as the dataset's event recipe resolves it. Scene cuts carry `cut_kind` (a change of scene or
a continuity cut) and the scene number. Recognition probes carry the patient's rating 1 to 6, the
response bucket `familiarity` (1 to 3 "novel", 4 to 6 "familiar") and `response_correct`.

**The truth is derived, not stored.** `familiarity` is the patient's answer, not whether the frame was
in the movie. A frame was old exactly when the answer and its correctness agree:
`truth = old` when (`familiarity == "familiar"`) equals (`response_correct == 1`).

In [ ]:
def task_events(
    session: asteroidea.Session,
) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """Return one run's scene cuts, recognition probes and movie interval."""
    events = session.events
    cuts = require(
        events.filter(pl.col("event_type") == "scene_cut").sort("onset_s"),
        f"scene cuts of {session.key}",
    ).select(
        pl.int_range(1, pl.len() + 1).alias("cut_number"),
        "onset_s",
        pl.col("attributes").str.json_path_match("$.cut_kind").alias("cut_kind"),
        pl.col("attributes")
        .str.json_path_match("$.scene_id")
        .cast(pl.Float64)
        .alias("scene_id"),
    )
    probes = (
        require(
            events.filter(pl.col("event_type") == "recognition_onset").sort("onset_s"),
            f"recognition probes of {session.key}",
        )
        .select(
            "onset_s",
            pl.col("attributes")
            .str.json_path_match("$.actual_response")
            .cast(pl.Float64)
            .cast(pl.Int64)
            .alias("rating"),
            pl.col("attributes")
            .str.json_path_match("$.familiarity")
            .alias("familiarity"),
            pl.col("attributes")
            .str.json_path_match("$.response_correct")
            .cast(pl.Float64)
            .eq(1)
            .alias("correct"),
        )
        .with_columns(
            pl.when((pl.col("familiarity") == "familiar") == pl.col("correct"))
            .then(pl.lit("old"))
            .otherwise(pl.lit("new"))
            .alias("truth")
        )
    )
    movie = require(
        events.filter(pl.col("event_type") == "movie_onset"),
        f"the movie of {session.key}",
    ).select(
        "onset_s",
        pl.coalesce(
            pl.col("stop_s"), pl.lit(float(cuts["onset_s"].to_numpy().max()))
        ).alias("stop_s"),
    )
    return cuts, probes, movie


task = {run: task_events(s) for run, s in sessions.items()}
show(
    pl.DataFrame(
        [
            {
                "run": RUN_LABEL[run],
                "movie_s": round(movie["stop_s"][0] - movie["onset_s"][0], 1),
                "change_cuts": cuts.filter(pl.col("cut_kind") == "change").height,
                "continuity_cuts": cuts.filter(
                    pl.col("cut_kind") == "continuity"
                ).height,
                "probes": probes.height,
                "new": probes.filter(pl.col("truth") == "new").height,
                "old": probes.filter(pl.col("truth") == "old").height,
                "correct": int(probes["correct"].sum()),
            }
            for run, (cuts, probes, movie) in task.items()
        ]
    )
)
show(
    pl.concat(
        [
            probes.with_columns(pl.lit(RUN_LABEL[run]).alias("run"))
            for run, (_, probes, _) in task.items()
        ]
    )
    .pivot(
        on="rating",
        index=["run", "truth"],
        values="onset_s",
        aggregate_function="len",
        sort_columns=True,
    )
    .fill_null(0)
    .sort("run", "truth")
)

## 3. Anatomy

Two views, in two different spaces, on purpose.

**The patient's own brain.** The OpenNeuro twin `sub-p62cs` has a T1-weighted MRI. `open_image()`
returns it as an image object read from the platform; `get_fdata()` gives the voxels and `affine` maps
voxels to scanner millimetres. This is **native space**: the patient's head as it lay in the scanner.
The slices go through the head's intensity centre of mass (sagittal, coronal) and about 90 mm below the
top of the head (axial, the level of the temporal lobes), cropped to the head.

In [ ]:
t1_files = require(
    pl.from_arrow(
        asteroidea.sql(
            "SELECT asset_id, s3_key, size_bytes FROM files "
            "WHERE subject = 'sub-p62cs' AND datatype = 'anat' AND suffix = 'T1w' "
            "AND s3_key LIKE '%run-1_T1w.nii.gz'"
        )
    ),
    "the run-1 T1w of sub-p62cs",
)
t1_image: Any = asteroidea.open_image(t1_files["asset_id"][0])
voxels = np.asarray(t1_image.get_fdata(), dtype=np.float32)
affine = np.asarray(t1_image.affine)
# Reorder the voxel axes to (left-right, back-front, down-up); flip backward ones
world_axis = np.abs(affine[:3, :3]).argmax(axis=0)
order = np.argsort(world_axis)
ras = np.transpose(voxels, order)
spacing = np.abs(affine[:3, :3])[world_axis[order], order]
for axis in range(3):
    if affine[axis, order[axis]] < 0:
        ras = np.flip(ras, axis=axis)
# The head: voxels above a tenth of the bright end; slices through its centre of mass
head = ras > 0.1 * np.percentile(ras, 99.5)
inside = [
    np.flatnonzero(head.any(axis=tuple(a for a in range(3) if a != k)))
    for k in range(3)
]
box = [slice(int(i[0]), int(i[-1]) + 1) for i in inside]
weights = np.where(head, ras, 0)
centre = [
    int(
        np.round(
            (
                weights.sum(axis=tuple(a for a in range(3) if a != k)) * np.arange(n)
            ).sum()
            / weights.sum()
        )
    )
    for k, n in enumerate(ras.shape)
]
# Axial cut about 90 mm below the top of the head: the level of the temporal lobes
centre[2] = max(int(inside[2][0]), int(inside[2][-1] - AXIAL_BELOW_TOP_MM / spacing[2]))
views = {
    "sagittal": (ras[centre[0], box[1], box[2]].T, spacing[2] / spacing[1]),
    "coronal": (ras[box[0], centre[1], box[2]].T, spacing[2] / spacing[0]),
    "axial": (ras[box[0], box[1], centre[2]].T, spacing[1] / spacing[0]),
}
fig, axes = plt.subplots(1, 3, figsize=(10, 3.8))
top = float(np.percentile(ras, 99.5))
for ax, (name, (plane, aspect)) in zip(axes, views.items(), strict=True):
    ax.imshow(plane, cmap="gray", origin="lower", vmin=0, vmax=top, aspect=aspect)
    ax.set_title(name)
    ax.set_axis_off()
fig.suptitle(
    f"{MRI_SUBJECT} T1w, native space: shape {voxels.shape}, "
    f"voxels {np.round(spacing, 2)} mm"
)
fig.tight_layout()
FIGURES["cs62-t1-native-slices.png"] = fig
plt.show()

**The electrodes on a standard brain.** The contact coordinates in the NWB files are in
MNI152NLin2009cAsym space (the paper's normalised space), not in the native space above, so they are
**not overlaid on the patient's T1**: drawing them there would place them wrongly. They are drawn on the
MNI152 template that ships with nilearn (no download), as a glass brain seen from the right, front
and top (every contact is in the right hemisphere), and on three slices through
the amygdala contacts. Colour is the area; the marker size grows with the number of units sorted on
that contact over both runs.

In [ ]:
unit_counts = units.group_by("electrode_index").agg(pl.len().alias("units"))
contacts = (
    electrodes.filter(pl.col("x").is_not_null() & (pl.col("area") != "other"))
    .unique(subset=["electrode_index"], keep="first", maintain_order=True)
    .join(unit_counts, on="electrode_index", how="left")
    .with_columns(pl.col("units").fill_null(0))
)
dropped = electrodes.unique(subset=["electrode_index"]).height - contacts.height
show(
    pl.DataFrame(
        {
            "contacts drawn": [contacts.height],
            "contacts without coordinates or area (not drawn)": [dropped],
        }
    )
)
# Every contact is in the right hemisphere: a right view, then coronal and axial
fig, (side, front) = plt.subplots(1, 2, figsize=(10, 3.6), width_ratios=(1, 2))
glasses = [
    plot_glass_brain(None, display_mode="r", axes=side, title="CS62 contacts, MNI152"),
    plot_glass_brain(None, display_mode="yz", axes=front),
]
for area in AREAS:
    picked = contacts.filter(pl.col("area") == area)
    if picked.is_empty():
        continue
    for glass in glasses:
        glass.add_markers(
            picked.select("x", "y", "z").to_numpy(),
            marker_color=AREA_COLOR[area],
            marker_size=(12 + 10 * picked["units"].to_numpy()).tolist(),
        )
fig.legend(
    handles=[
        Line2D([], [], marker="o", linestyle="", color=AREA_COLOR[a], label=a)
        for a in AREAS
    ],
    loc="lower center",
    ncol=len(AREAS),
)
FIGURES["cs62-contacts-glass-brain.png"] = fig
plt.show()

In [ ]:
amygdala_centre = (
    require(
        contacts.filter(pl.col("area") == "amygdala"),
        "amygdala contacts with coordinates",
    )
    .select("x", "y", "z")
    .mean()
    .row(0)
)
fig = plt.figure(figsize=(10, 3.6))
ortho = plot_anat(
    datasets.load_mni152_template(resolution=1),
    display_mode="ortho",
    cut_coords=amygdala_centre,
    figure=fig,
    title="MNI152 template, cut through the amygdala contacts",
    draw_cross=False,
    colorbar=False,
)
for area in AREAS:
    picked = contacts.filter(pl.col("area") == area)
    if not picked.is_empty():
        ortho.add_markers(
            picked.select("x", "y", "z").to_numpy(),
            marker_color=AREA_COLOR[area],
            marker_size=18,
        )
FIGURES["cs62-contacts-mni152-ortho.png"] = fig
plt.show()

## 4. Session overview

Every unit of a run over the whole session, as a heatmap: spike counts in 5 s bins, z-scored per unit
and clipped to +-3, rows grouped by area. Above the heatmap: the movie (shaded), the change cuts (red ticks) and continuity
cuts (grey ticks), and the recognition probes (blue for new, orange for old). The two runs sit side by
side. Most units change their rate between the movie and the recognition phase. A unit that is flat or
silent for a long stretch (some R1 amygdala units in the first minutes) is more likely a recording or
spike-sorting gap than biology: the paper notes that spikes can be missing in blocks after artifact
removal.

In [ ]:
fig, axes = plt.subplots(
    2, 2, figsize=(13, 8), height_ratios=(1, 7), sharex="col", layout="constrained"
)
heatmaps = []
for column, (run, session) in enumerate(sessions.items()):
    cuts, probes, movie = task[run]
    ordered = units.filter(pl.col("run") == RUN_LABEL[run]).sort(
        "area", "mean_rate_hz", descending=[False, True]
    )
    last_spikes = [float(t[-1]) for t in session.spikes.values() if t.size]
    end = max(float(probes["onset_s"].to_numpy().max()) + 5, *last_spikes)
    edges = np.arange(0, end, OVERVIEW_BIN_S)  # whole bins only
    counts = np.stack(
        [np.histogram(session.spikes[u], edges)[0] for u in ordered["unit_index"]]
    ).astype(float)
    image = np.clip(
        np.nan_to_num(zscore_rows(counts)), -OVERVIEW_Z_CLIP, OVERVIEW_Z_CLIP
    )
    timeline, heat = axes[0, column], axes[1, column]
    timeline.axvspan(movie["onset_s"][0], movie["stop_s"][0], color="#eeeeee")
    for kind, height in (("continuity", 0.5), ("change", 1.0)):
        at = cuts.filter(pl.col("cut_kind") == kind)["onset_s"].to_numpy()
        timeline.vlines(
            at, 0, height, color=CONDITION_COLOR[kind], linewidth=0.8, label=kind
        )
    for truth in ("new", "old"):
        at = probes.filter(pl.col("truth") == truth)["onset_s"].to_numpy()
        timeline.vlines(
            at,
            0,
            1,
            color=CONDITION_COLOR[truth],
            linewidth=1.2,
            label=f"{truth} probe",
        )
    timeline.set_yticks([])
    timeline.set_title(f"{RUN_LABEL[run]}: {ordered.height} units")
    heatmaps.append(
        heat.imshow(
            image,
            aspect="auto",
            cmap="RdBu_r",
            vmin=-OVERVIEW_Z_CLIP,
            vmax=OVERVIEW_Z_CLIP,
            extent=(edges[0], edges[-1], ordered.height, 0),
            interpolation="nearest",
        )
    )
    groups = (
        ordered.with_row_index("row")
        .group_by("area", maintain_order=True)
        .agg(pl.col("row").min().alias("first"), pl.len().alias("n"))
    )
    for first in groups["first"]:
        heat.axhline(first, color="white", linewidth=0.8)
    heat.set_yticks(
        (groups["first"] + groups["n"] / 2).to_list(),
        [f"{a} ({n})" for a, n in groups.select("area", "n").iter_rows()],
    )
    heat.set_xlabel("time in session (s)")
fig.colorbar(heatmaps[-1], ax=axes[1, :], label="z (clipped)", shrink=0.6)
axes[0, 1].legend(loc="upper left", bbox_to_anchor=(1.0, 1.4), fontsize=7)
fig.suptitle(
    f"Every unit over the whole session: {OVERVIEW_BIN_S:.0f} s bins, z-scored per unit"
)
FIGURES["cs62-session-overview.png"] = fig
plt.show()

How well did the patient remember? The rating against the truth for each run, and the ROC curve built
by sweeping the "old" criterion from 6 down to 1, with its area under the curve (AUC; 0.5 is chance,
1 is perfect). The paper keeps sessions whose AUC is at least 0.6.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), layout="constrained")
behaviour = {}
for column, (run, (_, probes, _)) in enumerate(task.items()):
    ratings = probes["rating"].to_numpy()
    is_old = (probes["truth"] == "old").to_numpy()
    false_alarms, hits, auc = roc_auc(ratings, is_old)
    behaviour[RUN_LABEL[run]] = auc
    counts = np.array(
        [
            [np.sum((ratings == r) & (is_old == o)) for r in range(1, 7)]
            for o in (False, True)
        ]
    )
    ax = axes[column]
    ax.imshow(counts, cmap="Blues")
    for (row, col), value in np.ndenumerate(counts):
        ax.text(col, row, str(value), ha="center", va="center", fontsize=8)
    ax.set_xticks(range(6), [str(r) for r in range(1, 7)])
    ax.set_yticks((0, 1), ("new", "old"))
    ax.set_xlabel("rating (1 sure new, 6 sure old)")
    ax.set_title(f"{RUN_LABEL[run]}: rating vs truth")
    axes[2].plot(
        false_alarms, hits, marker="o", label=f"{RUN_LABEL[run]} AUC {auc:.2f}"
    )
axes[2].plot((0, 1), (0, 1), color="grey", linestyle=":")
axes[2].set_xlabel("false alarm rate")
axes[2].set_ylabel("hit rate")
axes[2].set_title("ROC from the 1-6 ratings")
axes[2].legend()
show(
    pl.DataFrame(
        {"run": list(behaviour), "auc": [round(v, 3) for v in behaviour.values()]}
    )
)
FIGURES["cs62-recognition-behaviour.png"] = fig
plt.show()

## 5. Scene cuts (paper Fig 7)

**Question:** does a neuron fire differently after a cut that moves to a new scene than after a
continuity cut within the same scene? **Method,** as in the paper: each unit's rate in the 1 s after
every cut, change cuts against continuity cuts, compared with a bootstrap of the difference of means
(10,000 resamples, seed 13). A unit is **event selective** when p < 0.05 and it fires more after
changes. As in the release code, units that never fire during the movie are left out. Each unit is
tested on its own run's 93 cuts.

In [ ]:
def cut_test(run: str, unit: int) -> dict[str, Any]:
    """Return one unit's scene-cut test: rates per condition, difference and p."""
    cuts, _, _ = task[run]
    spikes = sessions[run].spikes[unit]
    change = window_rates(
        spikes,
        cuts.filter(pl.col("cut_kind") == "change")["onset_s"].to_numpy(),
        CUT_WINDOW_S,
    )
    continuity = window_rates(
        spikes,
        cuts.filter(pl.col("cut_kind") == "continuity")["onset_s"].to_numpy(),
        CUT_WINDOW_S,
    )
    return {
        "change_hz": float(change.mean()),
        "continuity_hz": float(continuity.mean()),
        "diff_hz": float(change.mean() - continuity.mean()),
        "p": bootstrap_p(change, continuity, balanced=False),
    }


def movie_rate(run: str, unit: int) -> float:
    """Return the unit's mean rate (Hz) during the movie."""
    _, _, movie = task[run]
    start, stop = float(movie["onset_s"][0]), float(movie["stop_s"][0])
    spikes = sessions[run].spikes[unit]
    return float(np.count_nonzero((spikes >= start) & (spikes < stop)) / (stop - start))


def pick_examples(results: pl.DataFrame) -> pl.DataFrame:
    """Return N_EXAMPLES units: selective ones by p, then the next best by p.

    The next best are units not called selective that fire at least
    MIN_EXAMPLE_RATE_HZ during the movie. A column `pick` says which.
    """
    chosen = results.filter(pl.col("selective")).sort("p").head(N_EXAMPLES)
    rest = (
        results.filter(
            ~pl.col("selective") & (pl.col("movie_rate_hz") >= MIN_EXAMPLE_RATE_HZ)
        )
        .sort("p")
        .head(N_EXAMPLES - chosen.height)
    )
    return require(
        pl.concat(
            [
                chosen.with_columns(pl.lit("selective").alias("pick")),
                rest.with_columns(pl.lit("next best").alias("pick")),
            ]
        ),
        f"example units (none selective, none at {MIN_EXAMPLE_RATE_HZ} Hz or more)",
    )


run_of = {RUN_LABEL[run]: run for run in RUNS}
tested = units.with_columns(
    pl.struct("run", "unit_index")
    .map_elements(
        lambda r: movie_rate(run_of[r["run"]], r["unit_index"]), return_dtype=pl.Float64
    )
    .alias("movie_rate_hz")
).filter(pl.col("movie_rate_hz") > 0)
started = time.monotonic()
cut_results = (
    tested.select("run", "unit_index", "area", "movie_rate_hz")
    .with_columns(
        pl.struct("run", "unit_index")
        .map_elements(
            lambda r: cut_test(run_of[r["run"]], r["unit_index"]),
            return_dtype=pl.Struct(
                {
                    "change_hz": pl.Float64,
                    "continuity_hz": pl.Float64,
                    "diff_hz": pl.Float64,
                    "p": pl.Float64,
                }
            ),
        )
        .alias("test")
    )
    .unnest("test")
    .with_columns(((pl.col("p") < ALPHA) & (pl.col("diff_hz") > 0)).alias("selective"))
)
show(
    pl.DataFrame(
        {
            "units tested": [cut_results.height],
            "silent in the movie (left out)": [units.height - tested.height],
            "event selective": [int(cut_results["selective"].sum())],
            "seconds": [round(time.monotonic() - started, 1)],
        }
    )
)
show(cut_results.sort("p").head(8))

Three example units: the event-selective units first (lowest p), and, if fewer than three are
selective, the next best by p among units that fire at least 0.5 Hz during the movie; each panel
says which. Each trial's spikes around the cut (change cuts in red on top, continuity cuts below on a
grey band) and the mean rate in 250 ms bins with its standard error, from 0.5 s before to 1 s after.

In [ ]:
top_cut = pick_examples(cut_results)
fig, axes = plt.subplots(
    2, 3, figsize=(12, 5.5), height_ratios=(2, 1), layout="constrained"
)
for column, (run_label, unit, area, p, pick) in enumerate(
    top_cut.select("run", "unit_index", "area", "p", "pick").iter_rows()
):
    cuts, _, _ = task[run_of[run_label]]
    draw_raster_psth(
        (axes[0, column], axes[1, column]),
        sessions[run_of[run_label]].spikes[unit],
        {
            kind: cuts.filter(pl.col("cut_kind") == kind)["onset_s"].to_numpy()
            for kind in ("change", "continuity")
        },
        CUT_PLOT_S,
    )
    axes[0, column].set_title(f"{run_label} unit {unit}, {area}: p = {p:.4f} ({pick})")
fig.suptitle("Scene cuts: example units")
FIGURES["cs62-scene-cut-examples.png"] = fig
plt.show()

The population: every tested unit's PSTH per condition, z-scored per unit over both conditions'
bins, then averaged over units (both runs pooled) with the standard error across units.

In [ ]:
def population_psth(
    results: pl.DataFrame, conditions: dict[str, str], window: tuple[float, float]
) -> dict[str, np.ndarray]:
    """Return each condition's per-unit z-scored PSTH, shape (units, bins).

    A unit's conditions are z-scored jointly.

    `conditions` maps a condition name to the column that selects it:
    `cut_kind` for scene cuts, `truth` for probes.
    """
    rows: dict[str, list[np.ndarray]] = {name: [] for name in conditions}
    for run_label, unit in results.select("run", "unit_index").iter_rows():
        run = run_of[run_label]
        cuts, probes, _ = task[run]
        per = {}
        for name, column in conditions.items():
            table = cuts if column == "cut_kind" else probes.filter(pl.col("correct"))
            onsets = table.filter(pl.col(column) == name)["onset_s"].to_numpy()
            per[name] = trial_psth(sessions[run].spikes[unit], onsets, window).mean(
                axis=0
            )
        joined = zscore_rows(np.concatenate(list(per.values()))[None, :])[0]
        if np.isnan(joined).any():
            continue
        for index, name in enumerate(conditions):
            size = per[name].size
            rows[name].append(joined[index * size : (index + 1) * size])
    return {name: np.stack(r) for name, r in rows.items()}


def draw_population(
    ax: plt.Axes, curves: dict[str, np.ndarray], window: tuple[float, float]
) -> None:
    """Draw each condition's mean z-scored PSTH with its SEM band."""
    centres = bin_centres(window)
    for name, rows in curves.items():
        mean, sem = mean_sem(rows)
        ax.plot(
            centres,
            mean,
            color=CONDITION_COLOR[name],
            label=f"{name} ({rows.shape[0]} units)",
        )
        ax.fill_between(
            centres, mean - sem, mean + sem, color=CONDITION_COLOR[name], alpha=0.25
        )
    ax.axvline(0, color="black", linewidth=0.7, linestyle="--")
    ax.set_xlabel("time from onset (s)")
    ax.set_ylabel("rate, z")
    ax.legend()


cut_population = population_psth(
    cut_results, {"change": "cut_kind", "continuity": "cut_kind"}, CUT_PLOT_S
)
fig, ax = plt.subplots(figsize=(5.5, 3.6), layout="constrained")
draw_population(ax, cut_population, CUT_PLOT_S)
ax.set_title("All units, both runs: response to scene cuts")
FIGURES["cs62-scene-cut-population.png"] = fig
plt.show()

Selective units per area, next to the paper's numbers. **The paper's values are cohort-level**: all
16 patients' units pooled, with an area-level permutation test this notebook does not run. One patient
cannot reproduce them; the comparison shows where CS62's units sit.

In [ ]:
PAPER_CUTS = pl.DataFrame(
    {
        "area": ["amygdala", "hippocampus", "preSMA", "ACC", "vmPFC"],
        "paper_selective": [15, 8, 20, 7, 10],
        "paper_units": [479, 261, 299, 236, 175],
        "paper_area_p": ["0.081", "0.199", "<0.001", "0.224", "0.005"],
    }
)
cut_by_area = (
    cut_results.group_by("area")
    .agg(
        pl.len().alias("cs62_units"), pl.col("selective").sum().alias("cs62_selective")
    )
    .join(PAPER_CUTS, on="area", how="full", coalesce=True)
    .with_columns(pl.col("cs62_units", "cs62_selective").fill_null(0))
    .with_columns(
        (100 * pl.col("cs62_selective") / pl.col("cs62_units"))
        .round(1)
        .alias("cs62_pct"),
        (100 * pl.col("paper_selective") / pl.col("paper_units"))
        .round(1)
        .alias("paper_pct"),
    )
    .sort("area")
)
show(cut_by_area)

## 6. Recognition (paper Fig 6)

**Question:** does a neuron fire differently to frames the patient has seen (old) than to new ones?
**Method,** as in the paper: correct trials only, each unit's rate 0.2 to 1.7 s after the frame
appears, new against old, bootstrap of the difference of means with both groups resampled to the
smaller group's size (10,000 resamples, seed 13). A unit is **memory selective** when p < 0.05, in
either direction: novelty units fire more to new frames, familiarity units to old ones.

In [ ]:
def probe_test(run: str, unit: int) -> dict[str, Any]:
    """Return one unit's new-vs-old test over correct trials."""
    _, probes, _ = task[run]
    correct = probes.filter(pl.col("correct"))
    spikes = sessions[run].spikes[unit]
    new = window_rates(
        spikes,
        correct.filter(pl.col("truth") == "new")["onset_s"].to_numpy(),
        PROBE_WINDOW_S,
    )
    old = window_rates(
        spikes,
        correct.filter(pl.col("truth") == "old")["onset_s"].to_numpy(),
        PROBE_WINDOW_S,
    )
    if new.size == 0 or old.size == 0:
        msg = f"{run} has no correct {'new' if new.size == 0 else 'old'} trials"
        raise LookupError(msg)
    return {
        "new_hz": float(new.mean()),
        "old_hz": float(old.mean()),
        "diff_hz": float(old.mean() - new.mean()),
        "p": bootstrap_p(old, new, balanced=True),
    }


started = time.monotonic()
probe_results = (
    tested.select("run", "unit_index", "area", "movie_rate_hz")
    .with_columns(
        pl.struct("run", "unit_index")
        .map_elements(
            lambda r: probe_test(run_of[r["run"]], r["unit_index"]),
            return_dtype=pl.Struct(
                {
                    "new_hz": pl.Float64,
                    "old_hz": pl.Float64,
                    "diff_hz": pl.Float64,
                    "p": pl.Float64,
                }
            ),
        )
        .alias("test")
    )
    .unnest("test")
    .with_columns(
        (pl.col("p") < ALPHA).alias("selective"),
        pl.when(pl.col("diff_hz") > 0)
        .then(pl.lit("familiarity"))
        .otherwise(pl.lit("novelty"))
        .alias("prefers"),
    )
)
show(
    probe_results.filter(pl.col("selective"))
    .group_by("prefers")
    .agg(pl.len().alias("units"))
    .with_columns(pl.lit(round(time.monotonic() - started, 1)).alias("seconds"))
)
show(probe_results.sort("p").head(8))

In [ ]:
top_probe = pick_examples(probe_results)
fig, axes = plt.subplots(
    2, 3, figsize=(12, 5.5), height_ratios=(2, 1), layout="constrained"
)
for column, (run_label, unit, area, p, pick) in enumerate(
    top_probe.select("run", "unit_index", "area", "p", "pick").iter_rows()
):
    _, probes, _ = task[run_of[run_label]]
    correct = probes.filter(pl.col("correct"))
    draw_raster_psth(
        (axes[0, column], axes[1, column]),
        sessions[run_of[run_label]].spikes[unit],
        {
            truth: correct.filter(pl.col("truth") == truth)["onset_s"].to_numpy()
            for truth in ("new", "old")
        },
        PROBE_PLOT_S,
    )
    axes[0, column].set_title(f"{run_label} unit {unit}, {area}: p = {p:.4f} ({pick})")
fig.suptitle("Recognition, correct trials: example units (same rule as for scene cuts)")
FIGURES["cs62-recognition-examples.png"] = fig
plt.show()

In [ ]:
probe_population = population_psth(
    probe_results, {"new": "truth", "old": "truth"}, PROBE_PLOT_S
)
fig, ax = plt.subplots(figsize=(5.5, 3.6), layout="constrained")
draw_population(ax, probe_population, PROBE_PLOT_S)
ax.set_title("All units, both runs: response to new and old frames")
PAPER_PROBES = pl.DataFrame(
    {
        "area": ["amygdala", "hippocampus", "preSMA", "ACC", "vmPFC"],
        "paper_selective": [32, 14, 17, 22, 14],
        "paper_units": [436, 238, 256, 186, 150],
        "paper_area_p": ["0.015", "0.30", "0.17", "0.002", "0.016"],
    }
)
probe_by_area = (
    probe_results.group_by("area")
    .agg(
        pl.len().alias("cs62_units"), pl.col("selective").sum().alias("cs62_selective")
    )
    .join(PAPER_PROBES, on="area", how="full", coalesce=True)
    .with_columns(pl.col("cs62_units", "cs62_selective").fill_null(0))
    .with_columns(
        (100 * pl.col("cs62_selective") / pl.col("cs62_units"))
        .round(1)
        .alias("cs62_pct"),
        (100 * pl.col("paper_selective") / pl.col("paper_units"))
        .round(1)
        .alias("paper_pct"),
    )
    .sort("area")
)
show(probe_by_area)
FIGURES["cs62-recognition-population.png"] = fig
plt.show()

## 7. Tuning

Beyond the two-condition tests: how does a unit's rate follow a graded variable?

**Rate against confidence.** For the memory-selective units (and, for scale, the mean over every
unit), the rate 0.2 to 1.7 s after each frame, averaged per rating 1 to 6 over all trials of the run.
Each unit's curve is z-scored across ratings so units with different rates share one axis.

In [ ]:
def rating_curve(run: str, unit: int) -> np.ndarray:
    """Return a unit's mean rate per rating 1..6 (NaN for a rating never given)."""
    _, probes, _ = task[run]
    rates = window_rates(
        sessions[run].spikes[unit], probes["onset_s"].to_numpy(), PROBE_WINDOW_S
    )
    ratings = probes["rating"].to_numpy()
    return np.array(
        [
            rates[ratings == r].mean() if np.any(ratings == r) else np.nan
            for r in range(1, 7)
        ]
    )


curves = {
    (run_label, unit): rating_curve(run_of[run_label], unit)
    for run_label, unit in probe_results.select("run", "unit_index").iter_rows()
}
selective_keys = [
    tuple(k)
    for k in probe_results.filter(pl.col("selective"))
    .select("run", "unit_index")
    .iter_rows()
]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
for key in selective_keys:
    axes[0].plot(range(1, 7), curves[key], marker="o", label=f"{key[0]} unit {key[1]}")
axes[0].set_xlabel("rating (1 sure new, 6 sure old)")
axes[0].set_ylabel("rate 0.2-1.7 s (Hz)")
axes[0].set_title(f"Memory-selective units ({len(selective_keys)})")
if selective_keys:
    axes[0].legend(fontsize=7)
stacked = np.stack([c for c in curves.values() if not np.isnan(c).all()])
z_curves = zscore_rows(np.nan_to_num(stacked, nan=np.nanmean(stacked)))
z_curves = z_curves[~np.isnan(z_curves).any(axis=1)]
mean, sem = mean_sem(z_curves)
axes[1].errorbar(range(1, 7), mean, yerr=sem, marker="o", color="black", capsize=3)
axes[1].set_xlabel("rating (1 sure new, 6 sure old)")
axes[1].set_ylabel("rate, z across ratings")
axes[1].set_title(f"Every unit, mean and SEM ({z_curves.shape[0]} units)")
FIGURES["cs62-rating-curves.png"] = fig
plt.show()

**Change against continuity as an index.** `Session.tuning()` runs the platform's tuning screen for one
event attribute: per unit, the mean rate per level of `cut_kind` and a selectivity index (the spread of
the level means relative to their sum), with a permutation test. Its numbers are an exploratory screen
and use a permutation test, not the paper's bootstrap; the histogram marks the units the bootstrap
above called event selective.

In [ ]:
tuning = pl.concat(
    [
        sessions[run]
        .tuning("scene_cut", "cut_kind")
        .select("unit_index", "location", "selectivity_index", "p", "screened")
        .with_columns(pl.lit(RUN_LABEL[run]).alias("run"))
        for run in RUNS
    ]
).join(
    cut_results.select("run", "unit_index", "selective"),
    on=["run", "unit_index"],
    how="left",
)
fig, ax = plt.subplots(figsize=(6, 3.6), layout="constrained")
bins = np.linspace(0, np.nanmax(tuning["selectivity_index"].to_numpy()), 25).tolist()
for flag, colour, label in (
    (False, "#bbbbbb", "not selective"),
    (True, CONDITION_COLOR["change"], "event selective"),
):
    values = tuning.filter(pl.col("selective").fill_null(value=False) == flag)[
        "selectivity_index"
    ].drop_nulls()
    ax.hist(
        values.to_numpy(),
        bins=bins,
        color=colour,
        label=f"{label} ({values.len()})",
        alpha=0.85,
    )
ax.set_xlabel("selectivity index, cut_kind (Session.tuning)")
ax.set_ylabel("units")
ax.legend()
show(
    tuning.group_by("run")
    .agg(pl.len().alias("units"), pl.col("screened").sum().alias("screened"))
    .sort("run")
)
FIGURES["cs62-cut-kind-selectivity.png"] = fig
plt.show()

**Rate across the movie, twice.** The patient watched the same movie in both runs, so the population's
response to each cut can be compared between runs. Each unit's rate in the second after every cut,
z-scored per unit across cuts, averaged over the run's units, against the cut's number in the movie;
R1 and R2 on one axis, change cuts marked. The Pearson correlation between the two runs' traces tests
whether the cut-by-cut response repeats; the table under the figure states the verdict. A correlation
near zero means no reliable cut-by-cut agreement at this resolution, not that the runs differ.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.8), layout="constrained")
traces = {}
for run, colour in zip(RUNS, ("#1b4f72", "#a04000"), strict=True):
    cuts, _, _ = task[run]
    onsets = cuts["onset_s"].to_numpy()
    keys = cut_results.filter(pl.col("run") == RUN_LABEL[run])["unit_index"].to_list()
    rows = zscore_rows(
        np.stack(
            [window_rates(sessions[run].spikes[u], onsets, CUT_WINDOW_S) for u in keys]
        )
    )
    rows = rows[~np.isnan(rows).any(axis=1)]
    mean, sem = mean_sem(rows)
    number = cuts["cut_number"].to_numpy()
    traces[run] = mean
    ax.plot(
        number,
        mean,
        color=colour,
        linewidth=1.2,
        label=f"{RUN_LABEL[run]} ({rows.shape[0]} units)",
    )
    ax.fill_between(number, mean - sem, mean + sem, color=colour, alpha=0.15)
for change_cut in task[RUNS[0]][0].filter(pl.col("cut_kind") == "change")["cut_number"]:
    ax.axvline(change_cut, color=CONDITION_COLOR["change"], linewidth=0.8, alpha=0.6)
if traces[RUNS[0]].shape != traces[RUNS[1]].shape:
    msg = f"the runs have different cut counts: {[t.size for t in traces.values()]}"
    raise ValueError(msg)
movie_r, movie_p = stats.pearsonr(traces[RUNS[0]], traces[RUNS[1]])
ax.set_xlabel("cut number in the movie (red lines: change cuts)")
ax.set_ylabel("rate 0-1 s after the cut, z")
movie_verdict = (
    "the runs agree cut by cut"
    if movie_p < ALPHA and movie_r > 0
    else "no reliable cut-by-cut agreement at this resolution"
)
ax.set_title(
    f"The same movie twice: R1 vs R2, Pearson r = {movie_r:.2f} (p = {movie_p:.2g})"
)
ax.legend()
FIGURES["cs62-cut-response-r1-vs-r2.png"] = fig
plt.show()
show(
    pl.DataFrame(
        {"pearson_r": [round(movie_r, 3)], "p": [movie_p], "verdict": [movie_verdict]}
    )
)

## 8. LFP (microwires)

The microwire LFP is a 1 kHz recording of every one of the 56 microwires. `Session.lfp()` reads a window
of an ElectricalSeries; here the window is the whole run. Its cost hardly depends on how many channels
are read (about 150 s per run either way), so each run is read once, all channels, in its
own cell. `times_s` come back on the same clock as the spikes and events (the reader applies the
recording's 10 s LFP offset). The samples are kept as float32, channels first, to halve the memory.

In [ ]:
def read_micro(run: str) -> tuple[np.ndarray, np.ndarray, float]:
    """Return one run's micro LFP: times, samples (channels, n) float32, rate."""
    lfp = sessions[run].lfp(MICRO_SERIES, None, start_s=0, duration_s=LFP_READ_S)
    if lfp.samples.shape[1] != N_MICRO or lfp.times_s.size < MIN_SAMPLES:
        msg = (
            f"{run}: micro LFP came back with shape {lfp.samples.shape}, "
            f"want (n, {N_MICRO})"
        )
        raise ValueError(msg)
    rate = float(round(1 / np.median(np.diff(lfp.times_s[:1000]))))
    return lfp.times_s, np.ascontiguousarray(lfp.samples.T, dtype=np.float32), rate


micro: dict[str, tuple[np.ndarray, np.ndarray, float]] = {}
started = time.monotonic()
micro[RUNS[0]] = read_micro(RUNS[0])
show(
    pl.DataFrame(
        {
            "run": ["R1"],
            "shape": [str(micro[RUNS[0]][1].shape)],
            "seconds": [round(time.monotonic() - started, 1)],
        }
    )
)

In [ ]:
started = time.monotonic()
micro[RUNS[1]] = read_micro(RUNS[1])
show(
    pl.DataFrame(
        {
            "run": ["R2"],
            "shape": [str(micro[RUNS[1]][1].shape)],
            "seconds": [round(time.monotonic() - started, 1)],
        }
    )
)

Each area's microwire bundle (the 8 microwires of that area, electrode indices 0 to 55) is cleaned:
60 Hz notches, 0.1 Hz high-pass, and the common average reference over the bundle's wires. The 16
RSPE microwires have no brain area and are left out. Then, per area and run, three products are kept
and the raw samples dropped, each checked to be finite with positive power:

- a whole-session spectrogram (2 s windows, half overlapping), averaged over the bundle's wires;
- the power around every scene cut and probe (0.256 s windows every 25 ms);
- the HFB trace, averaged over wires and reduced to 100 samples per second.

**Artifacts.** Microwire LFP carries large transients (movement, electrical or epileptiform) that a
mean over a few events cannot absorb. The paper does not state an artifact rule, so this notebook uses
a simple one: a sample is flagged when any wire of the bundle is beyond 6 robust standard deviations
(median absolute deviation x 1.4826) of its own median, or the HFB z is beyond 6. The common average is
zero by construction, so the wires are tested one by one. Each flag is padded by 0.25 s, and every
event whose plotted window (-0.5 to 1 s for cuts, -1 to 2 s for probes) touches a flag is dropped from
the event-locked spectrograms and HFB traces. The table lists the events kept (and the flagged seconds)
per area, run and condition.

In [ ]:
def micro_bundles(run: str) -> dict[str, list[int]]:
    """Return the micro channel positions of each area's bundle in one run.

    A bundle is every microwire (index below N_MICRO) of one area; RSPE has no area.
    """
    rows = require(
        electrodes.filter(
            (pl.col("bids_session") == run)
            & (pl.col("kind") == "micro")
            & pl.col("area").is_in(AREAS)
        ),
        f"micro electrodes of {run} in {AREAS}",
    )
    bundles: dict[str, list[int]] = {
        str(area): sorted(int(i) for i in group["electrode_index"])
        for (area,), group in rows.group_by("area", maintain_order=True)
    }
    small = {a: len(w) for a, w in bundles.items() if len(w) < MIN_BUNDLE_WIRES}
    if small:
        msg = (
            f"{run}: bundles with fewer than {MIN_BUNDLE_WIRES} wires {small}; "
            "a common average reference needs more"
        )
        raise ValueError(msg)
    return bundles


def lfp_products(run: str) -> dict[str, dict[str, Any]]:
    """Return per area: session spectrogram, event power and HFB trace of one run."""
    times, samples, rate = micro[run]
    cuts, probes, _ = task[run]
    events = {
        "change": cuts.filter(pl.col("cut_kind") == "change")["onset_s"].to_numpy(),
        "continuity": cuts.filter(pl.col("cut_kind") == "continuity")[
            "onset_s"
        ].to_numpy(),
        "new": probes.filter(pl.col("correct") & (pl.col("truth") == "new"))[
            "onset_s"
        ].to_numpy(),
        "old": probes.filter(pl.col("correct") & (pl.col("truth") == "old"))[
            "onset_s"
        ].to_numpy(),
    }
    windows = {
        "change": CUT_PLOT_S,
        "continuity": CUT_PLOT_S,
        "new": PROBE_PLOT_S,
        "old": PROBE_PLOT_S,
    }
    out: dict[str, dict[str, Any]] = {}
    for area, channels in micro_bundles(run).items():
        clean = clean_bundle(samples[channels], rate)
        label = f"{RUN_LABEL[run]} {area}"
        hfb = check_power(f"{label} HFB", hfb_trace(clean, rate), positive=False)
        mask = artifact_mask(clean, hfb, rate)
        kept = {
            name: clean_onsets(onsets, windows[name], mask, (float(times[0]), rate))
            for name, onsets in events.items()
        }
        freqs, seg_times, power = signal.spectrogram(
            clean,
            fs=rate,
            nperseg=round(SPEC_WINDOW_S * rate),
            noverlap=round(SPEC_WINDOW_S * rate / 2),
        )
        keep = (freqs >= 1) & (freqs <= SPEC_MAX_HZ)
        epochs = {
            name: epoch_power(clean, float(times[0]), rate, onsets)
            for name, onsets in kept.items()
        }
        for name, (_, _, event_power) in epochs.items():
            if event_power.shape[0]:
                check_power(f"{label} {name} event power", event_power, positive=True)
        out[area] = {
            "spec": (
                freqs[keep],
                seg_times + float(times[0]),
                check_power(
                    f"{label} spectrogram", power[:, keep].mean(axis=0), positive=True
                ),
            ),
            "epochs": epochs,
            "events": {name: onsets.size for name, onsets in events.items()},
            "kept": kept,
            "flags_1s": downsample(mask.astype(float), round(rate)) > 0,
            "hfb": downsample(hfb, round(rate / HFB_RATE_HZ)),
            "hfb_start_s": float(times[0]),
        }
    return out


started = time.monotonic()
lfp = {}
for run in RUNS:
    lfp[run] = lfp_products(run)
    micro.pop(run)
show(
    pl.DataFrame(
        {
            "run": [RUN_LABEL[r] for r in lfp],
            "areas": [", ".join(lfp[r]) for r in lfp],
            "seconds (both runs)": [round(time.monotonic() - started, 1)] * len(lfp),
        }
    )
)
show(
    pl.DataFrame(
        [
            {
                "area": area,
                "run": RUN_LABEL[run],
                "condition": name,
                "events": products["events"][name],
                "kept": int(onsets.size),
                "flagged_s": int(products["flags_1s"].sum()),
            }
            for run in RUNS
            for area, products in lfp[run].items()
            for name, onsets in products["kept"].items()
        ]
    )
    .pivot(on="condition", index=["area", "run", "flagged_s"], values="kept")
    .sort("area", "run")
)

**Whole-session spectrograms**, one row per area and one column per run, from 1 to 150 Hz. Raw LFP power
falls steeply with frequency, which hides everything else, so each frequency is shown in dB relative
to its own median over the session (a whitened spectrogram): red is more power than usual at that
frequency, blue less. The black strip at the top of each panel marks the artifact-flagged seconds. The
movie is between the dashed lines; the probes start after it.

In [ ]:
areas_present = [a for a in AREAS if all(a in lfp[r] for r in RUNS)]
fig, axes = plt.subplots(
    len(areas_present),
    2,
    figsize=(13, 2.2 * len(areas_present)),
    sharex="col",
    squeeze=False,
    layout="constrained",
)
meshes = []
for row, area in enumerate(areas_present):
    for column, run in enumerate(RUNS):
        freqs, seg_times, power = lfp[run][area]["spec"]
        ax = axes[row, column]
        white = 10 * np.log10(power / np.median(power, axis=1, keepdims=True))
        meshes.append(
            ax.pcolormesh(
                seg_times,
                freqs,
                white,
                shading="auto",
                cmap="RdBu_r",
                vmin=-WHITE_DB,
                vmax=WHITE_DB,
            )
        )
        flags = lfp[run][area]["flags_1s"]
        flag_clock = lfp[run][area]["hfb_start_s"] + np.arange(flags.size) + 0.5
        ax.fill_between(
            flag_clock,
            0.94,
            1.0,
            where=flags,
            transform=ax.get_xaxis_transform(),
            color="black",
            step="mid",
        )
        _, _, movie = task[run]
        for edge in (movie["onset_s"][0], movie["stop_s"][0]):
            ax.axvline(edge, color="white", linestyle="--", linewidth=0.8)
        ax.set_yscale("log")
        ax.set_ylim(1, SPEC_MAX_HZ)
        ax.set_ylabel(f"{area}\nHz")
        if row == 0:
            ax.set_title(RUN_LABEL[run])
    fig.colorbar(meshes[-1], ax=axes[row, :], label="dB re median", shrink=0.8)
for ax in axes[-1]:
    ax.set_xlabel("time in session (s)")
fig.suptitle("Micro LFP spectrogram per area, whitened (bundle mean, 2 s windows)")
FIGURES["cs62-lfp-session-spectrograms.png"] = fig
plt.show()

**Event-locked spectrograms**: the power around each clean event of both runs, as the median over events
(robust to a stray epoch) in dB change from the 0.5 s before the event. Columns: change cuts, continuity
cuts, new and old frames (correct trials). A panel with fewer than 5 clean events is left blank.

In [ ]:
conditions = ("change", "continuity", "new", "old")
fig, axes = plt.subplots(
    len(areas_present),
    4,
    figsize=(13, 2.2 * len(areas_present)),
    sharex=True,
    sharey=True,
    squeeze=False,
    layout="constrained",
)
meshes = []
for row, area in enumerate(areas_present):
    for column, name in enumerate(conditions):
        freqs, times = lfp[RUNS[0]][area]["epochs"][name][:2]
        power = np.concatenate([lfp[r][area]["epochs"][name][2] for r in RUNS])
        keep = (freqs >= EPOCH_MIN_HZ) & (freqs <= SPEC_MAX_HZ)
        ax = axes[row, column]
        if row == 0:
            ax.set_title(name)
        if column == 0:
            ax.set_ylabel(f"{area}\nHz")
        if power.shape[0] < MIN_CLEAN_EVENTS:
            ax.text(
                0.5,
                0.5,
                f"too few clean events ({power.shape[0]})",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            continue
        change = check_power(
            f"{area} {name} dB map", db_change(power[:, keep], times), positive=False
        )
        meshes.append(
            ax.pcolormesh(
                times,
                freqs[keep],
                change,
                shading="auto",
                cmap="RdBu_r",
                vmin=-3,
                vmax=3,
            )
        )
        ax.axvline(0, color="black", linewidth=0.7, linestyle="--")
        ax.text(0.02, 0.92, f"n = {power.shape[0]}", transform=ax.transAxes, fontsize=7)
for ax in axes[-1]:
    ax.set_xlabel("time from event (s)")
if meshes:
    fig.colorbar(meshes[-1], ax=axes, label="dB re -0.5 to 0 s (median)", shrink=0.6)
fig.suptitle("Event-locked micro LFP power, clean events of both runs")
FIGURES["cs62-lfp-event-spectrograms.png"] = fig
plt.show()

**High-frequency broadband (HFB)** by condition, the LFP measure the paper tests: 70 to 170 Hz power,
z-scored over the whole run per frequency band and wire, averaged, then cut around each clean event.
Mean and standard error over the clean events of both runs; a condition with fewer than 5 is not drawn.

In [ ]:
def hfb_epochs(
    run: str, area: str, onsets: np.ndarray, window: tuple[float, float]
) -> np.ndarray:
    """Return the HFB trace around each onset, shape (events, samples)."""
    trace = lfp[run][area]["hfb"]
    first = np.round(
        (onsets + window[0] - lfp[run][area]["hfb_start_s"]) * HFB_RATE_HZ
    ).astype(int)
    width = round((window[1] - window[0]) * HFB_RATE_HZ)
    inside = (first >= 0) & (first + width <= trace.size)
    if not np.any(inside):
        return np.zeros((0, width))
    return np.stack([trace[f : f + width] for f in first[inside]])


fig, axes = plt.subplots(
    len(areas_present),
    2,
    figsize=(10, 2.0 * len(areas_present)),
    sharex="col",
    squeeze=False,
    layout="constrained",
)
hfb_peaks = {}
for row, area in enumerate(areas_present):
    for column, (pair, window) in enumerate(
        ((("change", "continuity"), CUT_PLOT_S), (("new", "old"), PROBE_PLOT_S))
    ):
        ax = axes[row, column]
        notes = []
        for name in pair:
            epochs = np.concatenate(
                [hfb_epochs(r, area, lfp[r][area]["kept"][name], window) for r in RUNS]
            )
            if epochs.shape[0] < MIN_CLEAN_EVENTS:
                notes.append(f"{name}: too few clean events ({epochs.shape[0]})")
                continue
            mean, sem = mean_sem(epochs)
            clock = window[0] + np.arange(mean.size) / HFB_RATE_HZ
            ax.plot(
                clock,
                mean,
                color=CONDITION_COLOR[name],
                label=f"{name} ({epochs.shape[0]})",
            )
            ax.fill_between(
                clock, mean - sem, mean + sem, color=CONDITION_COLOR[name], alpha=0.25
            )
            hfb_peaks[f"{area} {name}"] = float(mean[clock >= 0].max())
        ax.axvline(0, color="black", linewidth=0.7, linestyle="--")
        ax.set_ylabel(f"{area}\nHFB, z")
        if notes:
            ax.text(0.02, 0.05, "; ".join(notes), transform=ax.transAxes, fontsize=7)
        ax.legend(fontsize=7, ncol=2)
for ax in axes[-1]:
    ax.set_xlabel("time from event (s)")
fig.suptitle("HFB (70-170 Hz) power by condition, clean events of both runs")
FIGURES["cs62-lfp-hfb-by-condition.png"] = fig
plt.show()

## 9. Summary

The headline numbers of this run in one table, collected in a `headline` dictionary. Every number above
is recomputed from the platform each time the notebook runs.

In [ ]:
headline: dict[str, Any] = {
    "patient": f"{SUBJECT} (DANDI 000623) = {MRI_SUBJECT} (OpenNeuro)",
    "participant key": patient_key,
    "runs": ", ".join(RUN_LABEL[r] for r in RUNS),
    "units": units.height,
    "units per run": json.dumps(
        dict(units.group_by("run").len().sort("run").iter_rows())
    ),
    "units tested (fire in the movie)": cut_results.height,
    "contacts with MNI coordinates": contacts.height,
    "scene cuts per run": json.dumps({RUN_LABEL[r]: task[r][0].height for r in RUNS}),
    "event-selective units (scene cuts)": int(cut_results["selective"].sum()),
    "memory-selective units (recognition)": int(probe_results["selective"].sum()),
    "novelty / familiarity units": json.dumps(
        dict(
            probe_results.filter(pl.col("selective"))
            .group_by("prefers")
            .len()
            .iter_rows()
        )
    ),
    "R1 vs R2 cut-response r (p)": f"{movie_r:.3f} (p {movie_p:.2g}): {movie_verdict}",
    "behavioural AUC": json.dumps({k: round(v, 3) for k, v in behaviour.items()}),
    "T1 shape": str(voxels.shape),
    "LFP areas": ", ".join(areas_present),
    "bootstrap": f"{BOOTSTRAP_N} resamples, seed {SEED}, alpha {ALPHA}",
}
show(
    pl.DataFrame(
        {"metric": list(headline), "value": [str(v) for v in headline.values()]}
    )
)

## 10. Save results

With `SAVE_RESULTS` on, the run opened in section 1 saves this notebook's outputs as platform results,
readable by every researcher in the web app's Results place: each figure as a PNG, `headline` as JSON,
and the scene-cut and recognition unit tables and their per-area tables as CSV. Each saved file carries
the run's record of what was read. Every run of the notebook adds a new batch, and the earlier batches
stay as history. The run is closed either way. The executed HTML report is not saved here: it exists
only after the notebook has finished.

The table lists each saved file and its page in the web app.

In [ ]:
SAVED_SCHEMA = {"name": pl.String, "page": pl.String, "bytes": pl.Int64}


def save_results(
    run: asteroidea.Run,
    figures: dict[str, Any],
    numbers: dict[str, Any],
    tables: dict[str, pl.DataFrame],
    *,
    enabled: bool,
) -> pl.DataFrame:
    """Save the figures (PNG), the numbers (JSON) and the tables (CSV) in `run`.

    Return one row per saved file: its name and its page in the web app. With
    `enabled` False nothing is saved and the one row says so.
    """
    if not enabled:
        return pl.DataFrame(
            {
                "name": ["saving off (SAVE_RESULTS is False)"],
                "page": [None],
                "bytes": [None],
            },
            schema=SAVED_SCHEMA,
        )
    folder = Path(tempfile.mkdtemp(prefix="showcase-results-"))
    for name, figure in figures.items():
        figure.savefig(folder / name, dpi=150, bbox_inches="tight")
    (folder / "cs62-headline.json").write_text(
        json.dumps(numbers, indent=2, default=str)
    )
    for name, table in tables.items():
        table.write_csv(folder / name)
    rows = []
    for name in [*figures, "cs62-headline.json", *tables]:
        result = run.save_artifact(folder / name, note=f"Showcase notebook, {SUBJECT}")
        rows.append(
            {
                "name": result.name,
                "page": f"/results/{result.artifact_id}",
                "bytes": result.size,
            }
        )
    return pl.DataFrame(rows, schema=SAVED_SCHEMA)


with run_scope:  # closes the run opened in section 1, after the saves
    saved = save_results(
        results_run,
        FIGURES,
        headline,
        {
            "cs62-scene-cut-units.csv": cut_results,
            "cs62-scene-cut-by-area.csv": cut_by_area,
            "cs62-recognition-units.csv": probe_results,
            "cs62-recognition-by-area.csv": probe_by_area,
        },
        enabled=SAVE_RESULTS,
    )
show(saved)